<a href="https://colab.research.google.com/github/Darksitomx/colab/blob/main/AI_Image_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Image Lab — ComfyUI en Google Colab

Notebook para ejecutar ComfyUI con GPU de Colab, persistir modelos/LoRAs/workflows en Google Drive y acceder a la interfaz desde el navegador.

**Arquitectura:** Colab GPU → ComfyUI → Google Drive (`AI-Image`) → modelos / LoRAs / outputs.


## 1. Configuración
En Colab selecciona **Runtime → Change runtime type → GPU** antes de ejecutar las celdas.

In [4]:
import os, subprocess, sys, time
from pathlib import Path

print('Entorno preparado.')


Entorno preparado.


In [5]:
!nvidia-smi

import torch
print('\nPyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')


Mon Sep 21 06:31:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Google Drive
Los modelos, LoRAs, workflows y resultados permanecerán en Drive aunque la sesión de Colab termine.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AI-Image')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

for folder in [
    'models/checkpoints',
    'models/loras',
    'models/vae',
    'models/controlnet',
    'models/upscale_models',
    'models/clip',
    'output',
    'input',
    'workflows',
]:
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

print('Persistencia:', DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistencia: /content/drive/MyDrive/AI-Image


## 3. Instalar ComfyUI
Usamos el repositorio oficial y dejamos el código de ComfyUI en `/content` para no hacer que cada operación de la interfaz dependa de la velocidad de Drive. Los modelos y resultados sí viven en Drive.

In [7]:
COMFY_DIR = Path('/content/ComfyUI')

if not (COMFY_DIR / 'main.py').exists():
    !git clone https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI

%cd /content/ComfyUI
!pip install -q -r requirements.txt

print('ComfyUI instalado en', COMFY_DIR)


/content/ComfyUI
ComfyUI instalado en /content/ComfyUI


## 4. Conectar modelos y resultados con Google Drive
Creamos enlaces simbólicos para que ComfyUI vea directamente las carpetas persistentes.

In [8]:
import shutil

MODEL_DIR = COMFY_DIR / 'models'
DRIVE_MODELS = DRIVE_ROOT / 'models'

for name in ['checkpoints','loras','vae','controlnet','upscale_models','clip']:
    target = MODEL_DIR / name
    source = DRIVE_MODELS / name
    source.mkdir(parents=True, exist_ok=True)
    if target.is_symlink() or target.exists():
        if target.is_symlink():
            target.unlink()
        elif target.is_dir():
            shutil.rmtree(target)
    target.symlink_to(source, target_is_directory=True)

output_dir = COMFY_DIR / 'output'
if output_dir.is_symlink():
    output_dir.unlink()
elif output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.symlink_to(DRIVE_ROOT / 'output', target_is_directory=True)

print('Modelos:', DRIVE_MODELS)
print('Outputs:', DRIVE_ROOT / 'output')


Modelos: /content/drive/MyDrive/AI-Image/models
Outputs: /content/drive/MyDrive/AI-Image/output


## 5. ComfyUI Manager
El Manager permite instalar y administrar custom nodes desde la interfaz.

In [9]:
MANAGER = COMFY_DIR / 'custom_nodes' / 'ComfyUI-Manager'
if not MANAGER.exists():
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
print('Manager listo.')


Manager listo.


## 6. Descarga opcional de modelos
Coloca URLs directas de archivos `.safetensors` en las listas. No se descarga ningún modelo automáticamente en esta versión para evitar llenar Drive accidentalmente.

In [19]:
MODEL_URLS = {
    'checkpoints': [
        "https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/flux1-dev.safetensors"
    ],
    'loras': [],
    'vae': [],
    'controlnet': [],
}

print('Añade URLs a MODEL_URLS cuando quieras automatizar descargas.')


Añade URLs a MODEL_URLS cuando quieras automatizar descargas.


In [ ]:
import urllib.request
from google.colab import userdata

def download_urls(category):
    urls = MODEL_URLS.get(category, [])
    destination = DRIVE_MODELS / category
    destination.mkdir(parents=True, exist_ok=True)

    # Intentar obtener el token de Hugging Face desde los secretos de Colab
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None

    for url in urls:
        filename = url.split('?')[0].rstrip('/').split('/')[-1]
        if not filename:
            raise ValueError(f'No se pudo determinar el nombre: {url}')
        path = destination / filename
        if path.exists():
            print('Ya existe:', path.name)
            continue
        print('Descargando:', filename)

        req = urllib.request.Request(url)
        if hf_token and "huggingface.co" in url:
            req.add_header('Authorization', f'Bearer {hf_token}')

        try:
            with urllib.request.urlopen(req) as response, open(path, 'wb') as out_file:
                out_file.write(response.read())
            print('Listo:', path)
        except Exception as e:
            print(f'Error al descargar {filename}: {e}')

for category in MODEL_URLS:
    download_urls(category)

Descargando: flux1-dev.safetensors


## 7. Lanzar ComfyUI
La interfaz escucha en el puerto `8188`.

In [15]:
%cd /content/ComfyUI

import subprocess, threading, time

if 'comfy_process' in globals() and comfy_process.poll() is None:
    print('ComfyUI ya está ejecutándose.')
else:
    comfy_process = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    def stream_logs():
        for line in comfy_process.stdout:
            print(line, end='')

    threading.Thread(target=stream_logs, daemon=True).start()
    time.sleep(5)
    print('ComfyUI iniciado.')


/content/ComfyUI
ComfyUI ya está ejecutándose.


## 8. URL pública con Cloudflare Tunnel
La URL es temporal y cambia al reiniciar la sesión.

In [16]:
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1 || true

import subprocess, re, time
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.25)
        continue
    print(line, end='')
    match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print('\nURL de ComfyUI:', url or 'No detectada; revisa los logs.')


2026-09-21T06:35:12Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-21T06:35:12Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-21T06:35:16Z INF +--------------------------------------------------------------------------------------------+
2026-09-21T06:35:16Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-21T06:35:16Z INF |  https://evanescence-crawford-tri-kijiji.trycloudflare

In [18]:
from pathlib import Path

p = Path("/content/ComfyUI/models/checkpoints/flux1-schnell.safetensors")

print("Existe:", p.exists())
print("Tamaño:", p.stat().st_size / (1024**3) if p.exists() else 0, "GB")

Existe: True
Tamaño: 0.0 GB


## 9. Estructura final
```text
Google Drive/MyDrive/AI-Image/
├── models/
│   ├── checkpoints/
│   ├── loras/
│   ├── vae/
│   ├── controlnet/
│   └── upscale_models/
├── input/
├── output/
└── workflows/
```

### Próximos módulos
- Selector de modelos desde una celda de configuración.
- Descarga segura de checkpoints/LoRAs desde fuentes concretas.
- Workflows preconfigurados para SD 1.5 y SDXL.
- Panel sencillo para prompt, seed, steps, CFG y LoRA weights.
- Detección automática de T4/L4/A100 y ajuste de memoria.
